## 2. Linear Regression Baseline

In [8]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [9]:
import json
import numpy as np

from src.dataloader import load_arrays, load_cgm_stats, load_participant_ids
from src.metrics import evaluate, format_results_table, evaluate_per_participant
from src.models.classical_baseline import LinearRegressionPredictor, HORIZONS

MODEL_NAME = 'LinearRegression'

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'linear_regression'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# store results in linear_regression
RESULTS_PATH = RESULTS_DIR / 'lr_results.json'
PARTICIPANT_SUMMARY_PATH = RESULTS_DIR / 'lr_participant_stability_summary.json'

In [10]:
stats = load_cgm_stats()
X_train, y_train = load_arrays('train')
X_val, y_val = load_arrays('val')
X_test, y_test = load_arrays('test')

test_participant_ids = load_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print('train set shape: ', X_train.shape, '\nvaluation set shape: ', X_val.shape, '\ntest set shape: ', X_test.shape)
print('\nnumber of test participants: ', len(PARTICIPANT_IDS))
print('Corresponding participants ID: \n', PARTICIPANT_IDS)

train set shape:  (83532, 24) 
valuation set shape:  (16739, 24) 
test set shape:  (16842, 24)

number of test participants:  6
Corresponding participants ID: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]


In [11]:
model = LinearRegressionPredictor().fit(X_train, y_train)

In [12]:
# Evaluate Linear Regression
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

val_pred = model.predict(X_val)
test_pred = model.predict(X_test)

val_results = evaluate(
    val_pred, y_val,
    stats['cgm_mean'], stats['cgm_std']
)

test_results = evaluate(
    test_pred, y_test,
    stats['cgm_mean'], stats['cgm_std']
)

test_results_per_participant = evaluate_per_participant(
    test_pred, y_test, test_participant_ids,
    stats['cgm_mean'], stats['cgm_std']
)

# Save complete results
result = {
    'model': MODEL_NAME,
    'seed': None,  # Linear Regression is deterministic
    'val_results': val_results,
    'test_results': test_results,
    'test_results_per_participant': test_results_per_participant,
}

with open(RESULTS_PATH, 'w') as f:
    json.dump(result, f, indent=2)

# Simple test-set output
print(f"\nTest results — {MODEL_NAME}\n")

for h in HORIZONS:
    r = test_results[h]
    zone_a = r['ceg']['A']
    zone_ab = r['ceg']['A'] + r['ceg']['B']

    print(
        f"{h:>3} min | "
        f"RMSE {r['rmse']:.2f} | "
        f"MAE {r['mae']:.2f} | "
        f"gRMSE {r['grmse']:.2f} | "
        f"Zone A {zone_a:.2f}% | "
        f"A+B {zone_ab:.2f}%"
    )

print(f"\n{'-' * 80}")
print(f"saved to {RESULTS_PATH}")


Test results — LinearRegression

 15 min | RMSE 11.57 | MAE 7.72 | gRMSE 12.82 | Zone A 96.54% | A+B 99.86%
 30 min | RMSE 19.25 | MAE 12.63 | gRMSE 22.43 | Zone A 89.88% | A+B 99.61%
 60 min | RMSE 28.14 | MAE 18.26 | gRMSE 34.55 | Zone A 81.20% | A+B 98.85%
 90 min | RMSE 33.62 | MAE 21.90 | gRMSE 42.16 | Zone A 75.24% | A+B 98.05%
120 min | RMSE 36.96 | MAE 24.21 | gRMSE 46.79 | Zone A 71.10% | A+B 97.33%

--------------------------------------------------------------------------------
saved to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/linear_regression/lr_results.json


In [13]:
def _extract(result, metric):
    if metric == 'zone_a':
        return result['ceg']['A']
    if metric == 'zone_ab':
        return result['ceg']['A'] + result['ceg']['B']
    return result[metric]


METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

participant_summary = {}

for h in HORIZONS:
    participant_summary[h] = {}

    for m in METRICS:

        # Main mean: pooled across all test samples
        mean = _extract(test_results[h], m)

        # SD1: no seed variability for deterministic Linear Regression
        sd1 = 0.0

        # SD2: variability across participants
        participant_values = np.array([
            _extract(test_results_per_participant[pid][h], m)
            for pid in PARTICIPANT_IDS
        ])

        sd2 = participant_values.std(ddof=1)

        participant_summary[h][m] = {
            'mean': float(mean),
            'sd1_across_seeds': float(sd1),
            'sd2_across_participants': float(sd2),
        }


print(
    f"=== {MODEL_NAME} test-set: mean +/- sd1 (sd2) | "
    f"sd1=0 (deterministic), sd2 across {len(PARTICIPANT_IDS)} participants ==="
)

for h in HORIZONS:
    s = participant_summary[h]

    print(
        f"{h:>3} min | "
        f"RMSE {s['rmse']['mean']:.2f} +/- {s['rmse']['sd1_across_seeds']:.2f} "
        f"({s['rmse']['sd2_across_participants']:.2f}) | "
        f"MAE {s['mae']['mean']:.2f} +/- {s['mae']['sd1_across_seeds']:.2f} "
        f"({s['mae']['sd2_across_participants']:.2f}) | "
        f"gRMSE {s['grmse']['mean']:.2f} +/- {s['grmse']['sd1_across_seeds']:.2f} "
        f"({s['grmse']['sd2_across_participants']:.2f}) | "
        f"Zone A {s['zone_a']['mean']:.2f} +/- {s['zone_a']['sd1_across_seeds']:.2f} "
        f"({s['zone_a']['sd2_across_participants']:.2f})% | "
        f"A+B {s['zone_ab']['mean']:.2f} +/- {s['zone_ab']['sd1_across_seeds']:.2f} "
        f"({s['zone_ab']['sd2_across_participants']:.2f})%"
    )


=== LinearRegression test-set: mean +/- sd1 (sd2) | sd1=0 (deterministic), sd2 across 6 participants ===
 15 min | RMSE 11.57 +/- 0.00 (2.03) | MAE 7.72 +/- 0.00 (1.49) | gRMSE 12.82 +/- 0.00 (3.23) | Zone A 96.54 +/- 0.00 (2.44)% | A+B 99.86 +/- 0.00 (0.21)%
 30 min | RMSE 19.25 +/- 0.00 (4.19) | MAE 12.63 +/- 0.00 (2.82) | gRMSE 22.43 +/- 0.00 (7.03) | Zone A 89.88 +/- 0.00 (3.69)% | A+B 99.61 +/- 0.00 (0.41)%
 60 min | RMSE 28.14 +/- 0.00 (9.63) | MAE 18.26 +/- 0.00 (6.00) | gRMSE 34.55 +/- 0.00 (15.10) | Zone A 81.20 +/- 0.00 (5.40)% | A+B 98.85 +/- 0.00 (1.24)%
 90 min | RMSE 33.62 +/- 0.00 (13.69) | MAE 21.90 +/- 0.00 (8.54) | gRMSE 42.16 +/- 0.00 (21.09) | Zone A 75.24 +/- 0.00 (8.26)% | A+B 98.05 +/- 0.00 (2.47)%
120 min | RMSE 36.96 +/- 0.00 (16.54) | MAE 24.21 +/- 0.00 (10.55) | gRMSE 46.79 +/- 0.00 (25.24) | Zone A 71.10 +/- 0.00 (9.39)% | A+B 97.33 +/- 0.00 (3.95)%


In [14]:
with open(PARTICIPANT_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': participant_summary,
    }, f, indent=2)

print(f"\nsaved to {PARTICIPANT_SUMMARY_PATH}")


saved to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/linear_regression/lr_participant_stability_summary.json
